# Example 16.8 / 1: Place Four Ornaments on the Source Tree

Companion notebook for **16.8 Christmas Tree Ornaments** in the AMPL lab Documentation, section "Place Four Ornaments on the Source Tree". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Place Four Ornaments on the Source Tree

The source notebook uses `height = 20`, `width = 0.4 * height = 8`, `sine_slope = 0.7`, `frequency = 1` and `offset = 0`, so `tree_slope = 5`. The run solves with the local solver Ipopt, reads `X` and `Y` with `ampl.get_data("X, Y").to_pandas()`, and validates the result in Python without reusing the model expressions:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
param n integer >= 2;  # number of ornaments
param width > 0;  # width of the tree base
param height > 0;  # height of the tree
param offset;  # vertical shift h0 of the garland curve
param frequency > 0;  # angular frequency omega of the sine
param sine_slope;  # linear slope a of the garland curve
param tree_slope := height / (width/2);  # slope of the tree's sides

set ORNAMENTS ordered := 1..n;  # ornaments in garland order
var X {ORNAMENTS} >= 0, <= width;  # horizontal position of each ornament
var Y {i in ORNAMENTS} =  # height on the garland curve, defined by X
    sin(frequency * X[i]) + sine_slope * X[i] + offset;
var min_sq_distance >= 0;  # smallest squared gap between neighbors

subject to InOrder {i in ORNAMENTS: ord(i) > 1}:  # keeps the garland order
    X[i] >= X[prev(i)];
subject to AboveBase {i in ORNAMENTS}:  # ornament above the tree base
    Y[i] >= 0;
subject to BelowLeftSide {i in ORNAMENTS}:  # ornament below the left side
    Y[i] <= tree_slope*X[i];
subject to BelowRightSide {i in ORNAMENTS}:  # ornament below the right side
    Y[i] <= tree_slope*(width-X[i]);
subject to Separate {i in ORNAMENTS: ord(i) > 1}:  # min_sq_distance bounds each neighbor gap
    min_sq_distance <=
        (X[i]-X[prev(i)])^2 + (Y[i]-Y[prev(i)])^2;

maximize Spacing: min_sq_distance;  # spread consecutive ornaments apart

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["n"] = 4
ampl.param["height"] = 20
ampl.param["width"] = 0.4 * 20
ampl.param["sine_slope"] = 0.7
ampl.param["frequency"] = 1
ampl.param["offset"] = 0

In [5]:
# run.py
import math

ampl.solve(solver="ipopt", verbose=False)
print(ampl.solve_result, round(ampl.obj["Spacing"].value(), 6))
sol = ampl.get_data("X, Y").to_pandas()
print(sol.round(4))
xs = list(sol["X"])
ys = [math.sin(x) + 0.7 * x for x in xs]
inside = all(-1e-6 <= y <= min(5 * x, 5 * (8 - x)) + 1e-6 for x, y in zip(xs, ys))
gaps = [(xs[i] - xs[i - 1]) ** 2 + (ys[i] - ys[i - 1]) ** 2 for i in range(1, 4)]
print("inside tree:", inside, "ordered:", xs == sorted(xs))
print("consecutive squared distances:", [round(g, 4) for g in gaps])

solved 9.889648
        X       Y
1  0.0000  0.0000
2  2.1070  2.3346
3  5.2201  2.7802
4  6.9141  5.4297
inside tree: True ordered: True
consecutive squared distances: [9.8896, 9.8896, 9.8896]


### Another way in amplpy

The smallest recomputed squared distance equals `Spacing`, and its square root is the ordinary minimum spacing.

The run below, with the same model.mod and data.py, solves the same tree with three ornaments and then moves the garland up with `offset = 30`:

In [6]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [7]:
# data.py
ampl.param["n"] = 4
ampl.param["height"] = 20
ampl.param["width"] = 0.4 * 20
ampl.param["sine_slope"] = 0.7
ampl.param["frequency"] = 1
ampl.param["offset"] = 0

In [8]:
# run.py
ampl.param["n"] = 3
ampl.solve(solver="ipopt", verbose=False)
print("n = 3:", ampl.solve_result, round(ampl.obj["Spacing"].value(), 6))
print("X:", [round(v, 4) for i, v in ampl.var["X"].to_list()])
ampl.param["offset"] = 30
ampl.solve(solver="ipopt", verbose=False)
print("offset = 30:", ampl.solve_result)

n = 3: solved 20.052108
X: [0, 3.985, 6.9141]
offset = 30: infeasible


### Pure AMPL: data.dat

The rows marked "Gurobi licence" are quoted from runs with Gurobi's global option, as in the source notebook; the others come from the runs on this page:

| Item | Concrete instance | Result |
|---|---|---|
| local solve | $n=3$, Ipopt | `solved`, `Spacing = 20.052108` |
| global solve with valid bounds (Gurobi licence) | $n=3$, Gurobi with `gurobi_options 'global=1 timelim=20'` | `solved`, `Spacing = 20.052108`, the value Ipopt returned |
| global check of the four-ornament example (Gurobi licence) | $n=4$, Gurobi global as above | `solved`, `Spacing = 9.889648`, the value Ipopt returned |
| curve outside the tree | `offset = 30` | Ipopt reports `infeasible`; by hand, $Y\ge30-1>20$, above the apex, so no point of the curve is inside |
| squared versus ordinary distance | `Spacing = 20.052108` | ordinary minimum spacing $\sqrt{20.052108}pprox4.478$ |

For pure AMPL, the four-ornament data in AMPL data format; with this data.dat, the first run above prints the same output. In pure AMPL, `option solver ipopt;`, `solve;` and `display X, Y;` do the same work, and Ipopt then also prints its banner and log:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
%%writefile data.dat
param n := 4;
param height := 20;
param width := 8;
param sine_slope := 0.7;
param frequency := 1;
param offset := 0;

Writing data.dat


In [11]:
%%ampl_eval
data data.dat;

In [12]:
# run.py
import math

ampl.solve(solver="ipopt", verbose=False)
print(ampl.solve_result, round(ampl.obj["Spacing"].value(), 6))
sol = ampl.get_data("X, Y").to_pandas()
print(sol.round(4))
xs = list(sol["X"])
ys = [math.sin(x) + 0.7 * x for x in xs]
inside = all(-1e-6 <= y <= min(5 * x, 5 * (8 - x)) + 1e-6 for x, y in zip(xs, ys))
gaps = [(xs[i] - xs[i - 1]) ** 2 + (ys[i] - ys[i - 1]) ** 2 for i in range(1, 4)]
print("inside tree:", inside, "ordered:", xs == sorted(xs))
print("consecutive squared distances:", [round(g, 4) for g in gaps])

solved 9.889648
        X       Y
1  0.0000  0.0000
2  2.1070  2.3346
3  5.2201  2.7802
4  6.9141  5.4297
inside tree: True ordered: True
consecutive squared distances: [9.8896, 9.8896, 9.8896]
